In [1]:
from pathlib import Path

folder = Path(r"D:\ACADS\4-2\FIN SOP\Data\Code\CrudeBert")
config_path = folder / "crude_bert_config.json"
model_path  = folder / "crude_bert_model.bin"

print("Config exists:", config_path.exists(), config_path)
print("Model exists :", model_path.exists(), model_path)

assert config_path.exists(), "crude_bert_config.json not found in the folder!"
assert model_path.exists(), "crude_bert_model.bin not found in the folder!"


Config exists: True D:\ACADS\4-2\FIN SOP\Data\Code\CrudeBert\crude_bert_config.json
Model exists : True D:\ACADS\4-2\FIN SOP\Data\Code\CrudeBert\crude_bert_model.bin


In [3]:
import torch
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

# Load config
config = AutoConfig.from_pretrained(str(config_path))

# Create model from config
model = AutoModelForSequenceClassification.from_config(config)

# Load model weights
state_dict = torch.load(str(model_path), map_location="cpu")

# Remove non-critical unexpected key if present
state_dict.pop("bert.embeddings.position_ids", None)

# Load into model
model.load_state_dict(state_dict, strict=False)

# Tokenizer (as per author instructions)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Loaded model + tokenizer ✅")
print("num_labels:", getattr(model.config, "num_labels", None))
print("id2label:", getattr(model.config, "id2label", None))


Loaded model + tokenizer ✅
num_labels: 3
id2label: {-1: 'negative', 0: 'neutral', 1: 'positive'}


In [4]:
# Author's stated label order:
# 0=positive, 1=negative, 2=neutral
model.config.id2label = {0: "positive", 1: "negative", 2: "neutral"}
model.config.label2id = {"positive": 0, "negative": 1, "neutral": 2}
model.config.num_labels = 3

class_names = ["positive", "negative", "neutral"]
print("Label mapping set ✅", model.config.id2label)


Label mapping set ✅ {0: 'positive', 1: 'negative', 2: 'neutral'}


In [71]:
def crudebert_raw_probs(headline: str):
    model.eval()

    inputs = tokenizer(
        headline,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].tolist()  # [p0, p1, p2]

    # Based on class_names order
    out = [
        {"label": class_names[i], "score": float(probs[i])}
        for i in range(len(class_names))
    ]
    return out

headline = "US Marines capture Venezualan president"
print("Headline:", headline)
print(crudebert_raw_probs(headline))


Headline: US Marines capture Venezualan president
[{'label': 'positive', 'score': 0.807804524898529}, {'label': 'negative', 'score': 0.18818813562393188}, {'label': 'neutral', 'score': 0.004007366951555014}]
